In [1]:
import numpy as np
import pandas as pd
import os
import glob
import re
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import importlib.util
import sys

In [2]:
DATA_DIR = "../cesnet-institutions-throughput/institutions/agg_1_hour_missing"
TRUE_DATASET = "../cesnet-institutions-throughput/institutions/agg_1_hour"
BASELINE_DIR = "./baseline"
OUTPUT_DIR = "./imputed_results"
EVALUATION_FILE = "./evaluation_results.csv"
PARTIAL_EVALUATION_FILE = "./resultados_parciais_avaliacao.csv"


os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
def load_imputation_functions():
    """Carrega todas as funções de imputação dos arquivos na pasta baselines e temporal_svd_knn"""
    functions = {}
    # Lista de diretórios para procurar por funções de imputação
    directories = [BASELINE_DIR, './temporal_svd_knn']
    
    for directory in directories:
        if not os.path.exists(directory):
            print(f"Diretório {directory} não encontrado. Pulando...")
            continue
            
        baseline_files = glob.glob(os.path.join(directory, "*.py"))
        
        for file_path in baseline_files:
            try:
                # Extrair nome do módulo
                module_name = os.path.splitext(os.path.basename(file_path))[0]
                
                # Carregar módulo
                spec = importlib.util.spec_from_file_location(module_name, file_path)
                module = importlib.util.module_from_spec(spec)
                sys.modules[module_name] = module
                spec.loader.exec_module(module)
                
                # Encontrar função de imputação (assumindo que começa com "impute_")
                for attr_name in dir(module):
                    if attr_name.startswith("impute_"):
                        functions[attr_name.replace("impute_", "")] = getattr(module, attr_name)
                        print(f"Carregada função: {attr_name} de {file_path}")
                
            except Exception as e:
                print(f"Erro ao carregar {file_path}: {e}")
    
    return functions

In [4]:
# results = []

# def evaluate_imputation(mask_missing, df_true, df_imp, method, file, rate):
#     y_true = df_true.loc[mask_missing, "throughput_bps"].values
#     y_pred = df_imp.loc[mask_missing, "throughput_bps"].values

#     if y_true.size == 0:
#         return results

#     rmse = np.sqrt(mean_squared_error(y_true, y_pred))
#     nrmse = rmse / (y_true.max() - y_true.min() + 1e-10)  # Evitar divisão por zero
#     nrmse_mean = rmse / (np.mean(y_true) + 1e-10)
#     mae = mean_absolute_error(y_true, y_pred)
#     mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100  # Evitar divisão por zero
#     r2 = r2_score(y_true, y_pred)

#     results.append({
#         "id_dataset": file,
#         "missing_rate": int(rate),
#         "imputation": method,
#         "rmse": rmse,
#         "nrmse": nrmse,
#         "nrmse_mean": nrmse_mean,
#         "mae": mae,
#         "mape": mape,
#         "r2": r2,
#     })
#     return results

def evaluate_imputation(mask_missing, df_true, df_imp, method, file, rate):
    y_true = df_true.loc[mask_missing, "throughput_bps"].values
    y_pred = df_imp.loc[mask_missing, "throughput_bps"].values

    if y_true.size == 0:
        return None

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    nrmse = rmse / (y_true.max() - y_true.min() + 1e-10)
    nrmse_mean = rmse / (np.mean(y_true) + 1e-10)
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100
    r2 = r2_score(y_true, y_pred)

    return {
        "id_dataset": file,
        "missing_rate": int(rate),
        "imputation": method,
        "rmse": rmse,
        "nrmse": nrmse,
        "nrmse_mean": nrmse_mean,
        "mae": mae,
        "mape": mape,
        "r2": r2,
    }

def save_partial_results(results, filename=PARTIAL_EVALUATION_FILE):
    """Salva resultados parciais em um arquivo CSV"""
    try:
        if results:
            df_partial = pd.DataFrame(results)
            # Se o arquivo já existe, adicionar ao existente
            if os.path.exists(filename):
                existing_df = pd.read_csv(filename)
                df_combined = pd.concat([existing_df, df_partial], ignore_index=True)
                df_combined.to_csv(filename, index=False)
            else:
                df_partial.to_csv(filename, index=False)
            print(f"Resultados parciais salvos em {filename}")
    except Exception as e:
        print(f"Erro ao salvar resultados parciais: {str(e)}")

In [5]:
# def process_datasets():
#     """Processa todos os datasets no diretório"""
#     # Inicializar lista de resultados
#     results = []
    
#     # Carregar funções de imputação
#     imputation_functions = load_imputation_functions()
#     if not imputation_functions:
#         print("Nenhuma função de imputação encontrada!")
#         return
    
#     # Encontrar todos os arquivos com padrão de nome
#     pattern = os.path.join(DATA_DIR, "*_missing_*.csv")
#     data_files = glob.glob(pattern)
    
#     if not data_files:
#         print(f"Nenhum arquivo encontrado com o padrão: {pattern}")
#         return
    
#     print(f"Encontrados {len(data_files)} arquivos para processar")
    
#     for file_path in data_files:
#         try:
#             # Extrair informações do nome do arquivo
#             filename = os.path.basename(file_path)
#             match = re.match(r"(.+)_missing_(\d+)\.csv", filename)
            
#             if not match:
#                 print(f"Padrão de nome inválido: {filename}")
#                 continue
                
#             id_dataset = match.group(1)
#             missing_rate = match.group(2)
            
#             print(f"Processando: {filename} (ID: {id_dataset}, Taxa: {missing_rate}%)")
            
#             # Carregar dataset
#             df_missing = pd.read_csv(file_path)
            
#             # Verificar se a coluna throughput_bps existe
#             if "throughput_bps" not in df_missing.columns:
#                 print(f"Coluna 'throughput_bps' não encontrada em {filename}")
#                 continue
                
#             # Criar máscara de valores missing
#             mask_missing = df_missing["throughput_bps"].isna()
            
#             # Carregar dados verdadeiros (assumindo que existe um arquivo sem missing)
#             true_file = os.path.join("../cesnet-institutions-throughput/institutions/agg_1_hour", f"{id_dataset}.throughput.csv")
#             if not os.path.exists(true_file):
#                 print(f"Arquivo verdadeiro não encontrado: {true_file}")
#                 continue
                
#             df_true = pd.read_csv(true_file)
            
#             # Aplicar cada método de imputação
#             for method_name, impute_func in imputation_functions.items():
#                 try:
#                     print(f"  Aplicando {method_name}...")
                    
#                     if impute_func.__name__ == 'impute_throughput_svd_knn':  # Match actual function name
#                         df_imputed = impute_func(
#                             df_missing,
#                             col="throughput_bps",
#                             min_period=24,
#                             max_period=1000,
#                             energy=0.9,
#                             k=10,
#                             allow_future=True
#                         )
#                     else:
#                         df_imputed = impute_func(df_missing.copy())
#                     # Salvar resultados
#                     output_file = os.path.join(OUTPUT_DIR, f"{id_dataset}_{missing_rate}_{method_name}.csv")
#                     df_imputed.to_csv(output_file, index=False)
                    
#                     # Avaliar imputação e adicionar aos resultados
#                     evaluation = evaluate_imputation(
#                         mask_missing, 
#                         df_true, 
#                         df_imputed, 
#                         method_name, 
#                         id_dataset, 
#                         missing_rate
#                     )
#                     if evaluation:
#                         results.append(evaluation)
                    
#                     print(f"    {method_name} concluído e salvo em {output_file}")
                    
#                 except Exception as e:
#                     print(f"    Erro ao aplicar {method_name}: {str(e)}")
#                     continue  # Continua para o próximo método
                    
#         except Exception as e:
#             print(f"Erro ao processar {file_path}: {str(e)}")
#             continue  # Continua para o próximo arquivo
    
#     # Salvar resultados da avaliação
#     if results:
#         df_results = pd.DataFrame(results)
#         df_results.to_csv(EVALUATION_FILE, index=False)
#         print(f"Resultados salvos em {EVALUATION_FILE}")
        
#         # Exibir resumo
#         print("\nResumo das avaliações:")
#         summary = df_results.groupby(["imputation", "missing_rate"]).mean()[["rmse", "mae", "r2"]]
#         print(summary)
#     else:
#         print("Nenhum resultado para salvar.")

# # Executar o processamento
# process_datasets()

def process_datasets(data_dir, partial_evaluation_file, evaluation_file, true_dataset, output_dir):
    """Processa todos os datasets no diretório"""
    # Inicializar lista de resultados
    results = []
    
    # Carregar funções de imputação
    imputation_functions = load_imputation_functions()
    if not imputation_functions:
        print("Nenhuma função de imputação encontrada!")
        return
    
    # Encontrar todos os arquivos com padrão de nome
    pattern = os.path.join(data_dir, "*_missing_*.csv")
    data_files = glob.glob(pattern)
    
    if not data_files:
        print(f"Nenhum arquivo encontrado com o padrão: {pattern}")
        return
    
    print(f"Encontrados {len(data_files)} arquivos para processar")
    
    # Verificar se há resultados parciais anteriores para continuar de onde parou
    if os.path.exists(partial_evaluation_file):
        try:
            existing_results = pd.read_csv(partial_evaluation_file)
            processed_files = existing_results["id_dataset"].unique()
            print(f"Encontrados {len(processed_files)} arquivos já processados anteriormente")
        except:
            processed_files = []
    else:
        processed_files = []
    
    for file_path in data_files:
        try:
            # Extrair informações do nome do arquivo
            filename = os.path.basename(file_path)
            match = re.match(r"(.+)_missing_(\d+)\.csv", filename)
            
            if not match:
                print(f"Padrão de nome inválido: {filename}")
                continue
                
            id_dataset = match.group(1)
            missing_rate = match.group(2)
            
            # Pular se já foi processado
            if id_dataset in processed_files:
                print(f"Pulando {filename} (já processado anteriormente)")
                continue
            
            print(f"Processando: {filename} (ID: {id_dataset}, Taxa: {missing_rate}%)")
            
            # Carregar dataset
            df_missing = pd.read_csv(file_path)
            
            # Verificar se a coluna throughput_bps existe
            if "throughput_bps" not in df_missing.columns:
                print(f"Coluna 'throughput_bps' não encontrada em {filename}")
                continue
                
            # Criar máscara de valores missing
            mask_missing = df_missing["throughput_bps"].isna()
            
            # Carregar dados verdadeiros
            true_file = os.path.join(true_dataset, f"{id_dataset}.throughput.csv")
            if not os.path.exists(true_file):
                print(f"Arquivo verdadeiro não encontrado: {true_file}")
                continue
                
            df_true = pd.read_csv(true_file)
            
            # Aplicar cada método de imputação
            for method_name, impute_func in imputation_functions.items():
                try:
                    print(f"  Aplicando {method_name}...")
                    
                    if impute_func.__name__ == 'impute_throughput_svd_knn':
                        df_imputed = impute_func(
                            df_missing,
                            col="throughput_bps",
                            min_period=24,
                            max_period=1000,
                            energy=0.9,
                            k=10,
                            allow_future=True
                        )
                    else:
                        df_imputed = impute_func(df_missing.copy())
                    
                    # Salvar resultados
                    output_file = os.path.join(output_dir, f"{id_dataset}_{missing_rate}_{method_name}.csv")
                    df_imputed.to_csv(output_file, index=False)
                    
                    # Avaliar imputação
                    evaluation = evaluate_imputation(
                        mask_missing, 
                        df_true, 
                        df_imputed, 
                        method_name, 
                        id_dataset, 
                        missing_rate
                    )
                    
                    if evaluation:
                        results.append(evaluation)
                        # Salvar resultados parciais após cada método
                        save_partial_results(results)
                        results = []  # Limpar lista após salvar
                    
                    print(f"    {method_name} concluído e salvo em {output_file}")
                    
                except Exception as e:
                    print(f"    Erro ao aplicar {method_name}: {str(e)}")
                    # Salvar resultados mesmo em caso de erro
                    if results:
                        save_partial_results(results)
                        results = []
                    continue
                    
        except Exception as e:
            print(f"Erro ao processar {file_path}: {str(e)}")
            # Salvar resultados mesmo em caso de erro
            if results:
                save_partial_results(results)
                results = []
            continue
    
    # Salvar resultados finais
    if results:
        save_partial_results(results)
    
    # Consolidar todos os resultados
    try:
        if os.path.exists(partial_evaluation_file):
            df_all_results = pd.read_csv(partial_evaluation_file)
            df_all_results.to_csv(evaluation_file, index=False)
            print(f"Resultados finais salvos em {evaluation_file}")
            
            # Exibir resumo
            print("\nResumo das avaliações:")
            summary = df_all_results.groupby(["imputation", "missing_rate"]).mean()[["rmse", "mae", "r2"]]
            print(summary)
    except Exception as e:
        print(f"Erro ao consolidar resultados finais: {str(e)}")



In [6]:
DATA_DIR = "../cesnet-institutions-throughput/institutions/agg_1_hour_missing"
TRUE_DATASET = "../cesnet-institutions-throughput/institutions/agg_1_hour"
BASELINE_DIR = "./baseline"
OUTPUT_DIR = "./imputed_results"
EVALUATION_FILE = "./evaluation_results.csv"
PARTIAL_EVALUATION_FILE = "./resultados_parciais_avaliacao.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Executar o processamento
process_datasets(DATA_DIR, PARTIAL_EVALUATION_FILE, EVALUATION_FILE, TRUE_DATASET, OUTPUT_DIR)



Carregada função: impute_arima de ./baseline\arima.py
Carregada função: impute_kalman de ./baseline\kalman_arima.py
Carregada função: impute_knn_imputer de ./baseline\knn.py
Carregada função: impute_linear_interpolation de ./baseline\linear_interpolation.py
Carregada função: impute_moving_average de ./baseline\moving_average.py
Carregada função: impute_pca de ./baseline\pca.py
Carregada função: impute_softimpute de ./baseline\soft_impute.py
Carregada função: impute_throughput_svd_knn de ./temporal_svd_knn\temporal_svd_knn.py
Encontrados 1132 arquivos para processar
Processando: 0_missing_10.csv (ID: 0, Taxa: 10%)
  Aplicando arima...
Resultados parciais salvos em ./resultados_parciais_avaliacao.csv
    arima concluído e salvo em ./imputed_results\0_10_arima.csv
  Aplicando kalman...
Resultados parciais salvos em ./resultados_parciais_avaliacao.csv
    kalman concluído e salvo em ./imputed_results\0_10_kalman.csv
  Aplicando knn_imputer...
Resultados parciais salvos em ./resultados_parc

In [7]:
DATA_DIR = "../cesnet-institutions-throughput/institutions/agg_6_hours_missing"
TRUE_DATASET = "../cesnet-institutions-throughput/institutions/agg_6_hours"
BASELINE_DIR = "./baseline"
OUTPUT_DIR = "./imputed_results_6_hours"
EVALUATION_FILE = "./evaluation_results_6_hours.csv"
PARTIAL_EVALUATION_FILE = "./resultados_parciais_avaliacao_6_hours.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Executar o processamento
process_datasets(DATA_DIR, PARTIAL_EVALUATION_FILE, EVALUATION_FILE, TRUE_DATASET, OUTPUT_DIR)



Carregada função: impute_arima de ./baseline\arima.py
Carregada função: impute_kalman de ./baseline\kalman_arima.py
Carregada função: impute_knn_imputer de ./baseline\knn.py
Carregada função: impute_linear_interpolation de ./baseline\linear_interpolation.py
Carregada função: impute_moving_average de ./baseline\moving_average.py
Carregada função: impute_pca de ./baseline\pca.py
Carregada função: impute_softimpute de ./baseline\soft_impute.py
Carregada função: impute_throughput_svd_knn de ./temporal_svd_knn\temporal_svd_knn.py
Encontrados 1132 arquivos para processar
Processando: 0_missing_10.csv (ID: 0, Taxa: 10%)
  Aplicando arima...
Resultados parciais salvos em ./resultados_parciais_avaliacao.csv
    arima concluído e salvo em ./imputed_results_6_hours\0_10_arima.csv
  Aplicando kalman...
Resultados parciais salvos em ./resultados_parciais_avaliacao.csv
    kalman concluído e salvo em ./imputed_results_6_hours\0_10_kalman.csv
  Aplicando knn_imputer...
Resultados parciais salvos em .